In [2]:
import torch 
import  numpy as np

In [ ]:
class Layer:
    def __init__(self,n_neurons,dim,activation):
        self.n_neurons=n_neurons
        self.activation=activation
        self.dim=dim
        self.weights=np.random.rand(n_neurons,dim)
        self.bias=np.ones(n_neurons)


    def forward(self,inp):
        self.inp=inp
        self.z=np.dot(self.weights,self.inp)+self.bias
        if self.activation=="ReLU":
            self.y=np.maximum(0,self.z)
        if self.activation=="Sigmoid":
            self.y=1/(1+np.exp(-self.z))
        if self.activation=="Linear":
            self.y=self.z
        return self.y
    
    def backward(self,grad):
        if self.activation == "ReLU":
            grad = grad * (self.z > 0)
        self.wgr=np.outer(grad,self.inp)
        self.bgr=grad
        dinput=self.weights.T @ grad
        self.weights=self.weights-0.05*self.wgr
        self.bias=self.bias-0.05*self.bgr
        return dinput

In [ ]:
class CNN:
    def __init__(self,h,w,c,n,m):
        self.h=h
        self.w=w
        self.c=c
        self.kernel_size=n
        self.pool_size=m
        self.filter=np.random.rand(n,n)
        self.layers=[]

    def add_layer(self, layer):
        self.layers.append(layer)

    def convolution(self,inp,outH,outW):
        res=np.zeros((outH,outW))
        for i in range(outH):
            for j in range(outW):
                for k in range(self.filter.shape[0]):
                    for m in range(self.filter.shape[1]):
                        res[i][j]+=self.filter[k][m]*inp[i+k][j+m]
        return res  
    
    def MaxPool(self,outH,outW,inp):
        res=np.zeros((outH,outW))
        ind=np.zeros((outH,outW,2),dtype=int)
        for i in range(outH):
            for j in range(outW):
                for k in range(self.pool_size):
                    for m in range(self.pool_size):
                        if(res[i][j]<inp[i+k][j+m]):
                            res[i][j]=inp[i+k][j+m]
                            ind[i][j]=[i+k,j+m]
        self.ind=ind
        return res
    
    def flatten(self,inp):
        return inp.reshape(-1)
    
    def forward(self,inp):
        self.inp=inp
        c_outH=inp.shape[0]-self.kernel_size+1
        c_outW=inp.shape[1]-self.kernel_size+1
        self.conv=self.convolution(self.inp,c_outH,c_outW)
        self.relu=np.maximum(0,self.conv)
        p_outH=c_outH-self.pool_size+1
        p_outW=c_outW-self.pool_size+1
        self.pool=self.MaxPool(p_outH,p_outW,self.relu)
        self.flat=self.flatten(self.pool)
        logits=self.flat

        for layer in self.layers:
            logits=layer.forward(logits)
        return logits
    
    def softmax(self,logits):
        logits-=np.max(logits)
        exp=np.exp(logits)
        self.prob=exp/np.sum(exp)
        return self.prob
    
    def CrossEntropy(self,logits,label):
        pred=self.softmax(logits)
        pred = np.clip(pred, 1e-15, 1 - 1e-15)
        self.loss=-np.log(pred[label])
        return self.loss
    
    def backward(self, label):

        # Softmax + CE
        target = np.zeros_like(self.prob)
        target[label] = 1
        grad = self.prob - target

        # Dense layers
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

        # Flatten
        grad = grad.reshape(self.pool.shape)

        # MaxPool
        grad = self.backprop_pool(grad)

        # ReLU
        grad = grad * (self.conv > 0)

        # Convolution
        self.backprop_conv(grad)

    def backprop_pool(self,grad):
        dpool=np.zeros_like(self.relu)
        for i in range(self.ind.shape[0]):
            for j in range(self.ind.shape[1]):
                dpool[self.ind[i][j][0]][self.ind[i][j][1]]+=grad[i][j]
        return dpool
    
    def backprop_relu(self,grad):
        return grad*(self.conv>0)
    
    def backprop_conv(self,grad):
        for i in range(self.kernel_size):
            for j in range(self.kernel_size):
                dw=0
                for k in range(grad.shape[0]):
                    for m in range(grad.shape[1]):
                        dw+=(self.inp[i+k][j+m]*grad[k][m])
                self.filter[i][j]-=0.05*dw
                        
    def train(self,inp,label):
        #convolution:
        result=self.forward(inp)
        loss=self.CrossEntropy(result,label)
        self.backward(label)

        return loss